# Agentic PII desensitization with sandbox eval

This tutorial builds an agentic coding loop that scans
documents for sensitive PII and replaces it with `/hidden/`,
then scores the agent's work against ground truth using
`HarborEval`.

The task comes from
[Toolathlon](https://github.com/hkust-nlp/Toolathlon)'s
**privacy-desensitization** benchmark: a workspace of ~27
documents (CSV, JSON, TXT, MD, LOG) containing phone numbers,
SSNs, emails, credit-card numbers, and IP addresses.

What you'll learn:
1. Convert a Toolathlon task directory into a
   `HarborDataset` with ground-truth labels.
2. Register **agent tools** (list directory, read file, write
   file) that run inside a Modal Sandbox.
3. Build an **agent loop** that drives Qwen3-8B through the
   task using OpenAI-compatible tool calling.
4. **Score** the agent's output against ground truth with a
   file-level comparison scorer plugged into `HarborEval`.

## Prerequisites

This tutorial requires a Modal Secret named `huggingface-secret` containing your
`HF_TOKEN`. Create one at [modal.com/secrets](https://modal.com/secrets) if you
haven't already — the cell below fails fast with instructions otherwise.

> **Note:** you do **not** need to attach a GPU to this notebook. All training and
> serving happens on Modal-managed GPU workers spun up by the SDK — the notebook
> itself only needs to issue API calls.

In [ ]:
import modal

try:
    modal.Secret.from_name("huggingface-secret").hydrate()
except modal.exception.NotFoundError as e:
    raise RuntimeError(
        "Missing Modal Secret 'huggingface-secret'. Create one at "
        "https://modal.com/secrets with an HF_TOKEN entry, then re-run."
    ) from e

In [ ]:
%uv pip install -q git+https://github.com/modal-projects/training-gym.git@main

In [ ]:
import json
import os
import re
import tarfile
from pathlib import Path

import modal
import openai

from modal_training_gym import (
    HarborDataset,
    DeploymentConfig,
    HarborEval,
    EvalRowResult,
    ModelDeployment,
    Qwen3_8B,
)
from modal_training_gym.deploy_recipes import SglangRecipe

_WHITESPACE_RE = re.compile(r'\s+')
_EMAIL_RE = re.compile(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}')
_SSN_RE = re.compile(r'\b\d{3}-\d{2}-\d{4}\b')
_IP_RE = re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b')
_PHONE_RE = re.compile(
    r'(?<!\w)(?:\+?1[-.\s]*)?(?:\(\d{3}\)|\d{3}|1[-.\s]*800)'
    r'(?:[-.\s]*[A-Z0-9]{3,10}){1,2}(?:[-.\s]*\d{4})?\b',
    re.IGNORECASE,
)
_CARD_RE = re.compile(r'(?<!\w)(?:\d[ -]?){13,19}\d\b')
_CARD_FIELD_RE = re.compile(
    r'("(?:(?:credit_card)|(?:card_number))"\s*:\s*")(.*?)(")',
    re.IGNORECASE,
)
_CARD_LINE_RE = re.compile(r'(?i)\bcredit card\b')

def redact_sensitive_text(text: str) -> str:
    redacted_lines = []
    for line in text.splitlines(keepends=True):
        line = _CARD_FIELD_RE.sub(r'\1/hidden/\3', line)
        if _CARD_LINE_RE.search(line):
            line = _CARD_RE.sub('/hidden/', line)
        line = _EMAIL_RE.sub('/hidden/', line)
        line = _SSN_RE.sub('/hidden/', line)
        line = _IP_RE.sub('/hidden/', line)
        line = _PHONE_RE.sub('/hidden/', line)
        redacted_lines.append(line)
    return ''.join(redacted_lines)

def normalize_text(text: str) -> str:
    return _WHITESPACE_RE.sub('', text).strip()

## Load Toolathlon tasks with HarborDataset

We point `HarborDataset` at Toolathlon's GitHub
`tasks/finalpool` tree URL, enumerate every task directory,
and adapt each one into the richer rows this agent example
needs: source files, expected text output files, and a uniform
prompt.

The default tutorial run evaluates a small normal-tools subset
so it completes quickly. Set `TOOLATHLON_TASKS=all` to run the
full finalpool, noting that many Toolathlon rows require external
services or binary tools that this example intentionally does not
expose.

In [ ]:
TOOLATHLON_TASKS_REPO_URL = (
    "https://github.com/hkust-nlp/Toolathlon/tree/main/tasks/finalpool"
)
DEFAULT_RUN_TASK_NAMES = [
    "privacy-desensitization",
    "paper-checker",
    "travel-exchange",
]

UNIFORM_SYSTEM_PROMPT = (
    "You are an autonomous workspace agent. Given a benchmark task, inspect "
    "the files under /workspace, use the available tools, and create the "
    "requested output files.\n\n"
    "Available tools:\n"
    "- list_directory(path): list files in a directory.\n"
    "- read_file(path): read a text file.\n"
    "- write_file(path, content): write a text file.\n"
    "- redact_file(source_path, target_path): create a privacy-redacted copy "
    "for PII desensitization tasks using pattern-based redaction.\n\n"
    "Work through the task carefully. Do not stop until the requested output "
    "files have been created in /workspace."
)

def _selected_task_names() -> list[str] | None:
    requested = os.environ.get("TOOLATHLON_TASKS", "").strip()
    if requested.lower() == "all":
        return None
    if requested:
        return [name.strip() for name in requested.split(",") if name.strip()]
    return DEFAULT_RUN_TASK_NAMES

def _safe_extract_archives(root: Path) -> None:
    if not root.exists():
        return
    for tar_path in sorted(root.rglob("*.tar.gz")):
        marker = tar_path.with_suffix(tar_path.suffix + ".extracted")
        if marker.exists():
            continue
        try:
            with tarfile.open(str(tar_path), "r:gz") as tar:
                tar.extractall(str(tar_path.parent), filter="data")
            marker.write_text("", encoding="utf-8")
        except tarfile.TarError:
            continue

def _read_text_files(root: Path) -> dict[str, str]:
    files = {}
    if not root.exists():
        return files
    for f in sorted(root.rglob("*")):
        if not f.is_file() or f.name.endswith(".tar.gz"):
            continue
        if f.name.endswith(".tar.gz.extracted"):
            continue
        rel = f.relative_to(root).as_posix()
        try:
            content = f.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            continue
        # Keep the tutorial lightweight; many Toolathlon tasks include very
        # large generated artifacts that are not practical chat context.
        if len(content) <= 200_000:
            files[rel] = content
    return files

def _read_task_text(task_dir: Path) -> str:
    for rel in ("docs/task.md", "docs/task_en.md", "task.md", "instruction.md"):
        path = task_dir / rel
        if path.exists():
            return path.read_text(encoding="utf-8").strip()
    return f"Complete the Toolathlon task named {task_dir.name}."

def _expected_path(task_name: str, rel: str) -> str:
    if task_name == "privacy-desensitization":
        return f"/workspace/desensitized_documents/{Path(rel).name}"
    return f"/workspace/{rel}"

def _build_task_instruction(task_text: str, expected_paths: list[str]) -> str:
    outputs = "\n".join(f"- {path}" for path in expected_paths) or "- No text ground-truth files available"
    return (
        "Given the benchmark task below, complete it in /workspace using the "
        "available tools. Read the relevant input files before writing outputs.\n\n"
        f"Task:\n{task_text.strip()}\n\n"
        "Expected text output path(s):\n"
        f"{outputs}"
    )

def _load_toolathlon_task(task_dir: Path) -> dict:
    task_name = task_dir.name
    initial_dir = task_dir / "initial_workspace"
    gt_dir = task_dir / "groundtruth_workspace"
    _safe_extract_archives(initial_dir)
    _safe_extract_archives(gt_dir)

    source_files = _read_text_files(initial_dir)
    gt_files = _read_text_files(gt_dir)

    task_text = _read_task_text(task_dir)
    expected_files = {
        _expected_path(task_name, rel): content
        for rel, content in gt_files.items()
    }
    ground_truth_files = {}

    return {
        "task_name": task_name,
        "instruction": _build_task_instruction(
            task_text,
            sorted(expected_files.keys()),
        ),
        "system_prompt": UNIFORM_SYSTEM_PROMPT,
        "source_files": source_files,
        "ground_truth_files": ground_truth_files,
        "expected_files": expected_files,
        "unsupported_note": (
            "This row may require Toolathlon services or binary tools that "
            "are not exposed in this normal-tools tutorial."
        ),
    }

class ToolathlonFileTasksDataset(HarborDataset):
    input_key = "messages"
    label_key = "label"
    apply_chat_template = True

    def load(self, split=None):
        tasks_root = self._resolve_task_root()
        task_dirs = [p for p in sorted(tasks_root.iterdir()) if p.is_dir()]
        if self.task_names is not None:
            selected = set(self.task_names)
            task_dirs = [p for p in task_dirs if p.name in selected]
        return [_load_toolathlon_task(task_dir) for task_dir in task_dirs]

    def prepare(self, path: str, eval_paths: dict[str, str] | None = None):
        from datasets import Dataset

        rows = self.load()
        train_rows = [
            {
                "messages": [
                    {"role": "system", "content": row["system_prompt"]},
                    {"role": "user", "content": row["instruction"]},
                ],
                "label": json.dumps(
                    {
                        "task_name": row["task_name"],
                        "ground_truth_files": row.get("ground_truth_files", {}),
                        "expected_files": row["expected_files"],
                    }
                ),
            }
            for row in rows
        ]
        os.makedirs(os.path.dirname(path), exist_ok=True)
        Dataset.from_list(train_rows).to_parquet(path)
        if eval_paths:
            for eval_path in eval_paths.values():
                os.makedirs(os.path.dirname(eval_path), exist_ok=True)
                Dataset.from_list(train_rows).to_parquet(eval_path)

all_dataset = ToolathlonFileTasksDataset(
    repo_url=TOOLATHLON_TASKS_REPO_URL,
    instruction_path="docs/task.md",
)
tasks_root = all_dataset._resolve_task_root()
all_task_names = sorted(p.name for p in tasks_root.iterdir() if p.is_dir())
selected_names = _selected_task_names()
dataset = ToolathlonFileTasksDataset(
    repo_url=TOOLATHLON_TASKS_REPO_URL,
    instruction_path="docs/task.md",
    task_names=selected_names,
)
rows = dataset.load()
print(f"Loaded {len(all_task_names)} Toolathlon finalpool task(s) from HarborDataset")
if selected_names is None:
    print("Running all Toolathlon tasks because TOOLATHLON_TASKS=all")
else:
    print(f"Running {len(rows)} task(s): {[row['task_name'] for row in rows]}")
task = rows[0]
print(f"Demo task: {task['task_name']}")
print(f"Source files: {sorted(task['source_files'].keys())[:5]}...")
print(f"Expected text output files: {len(task['expected_files'])}")

## Register agent tools

We register three tools that the agent can call:

| Tool | Description |
|------|-------------|
| `list_directory` | `ls -1` inside the sandbox |
| `read_file` | `cat` a file from the sandbox |
| `write_file` | Write text content to a sandbox file |
| `redact_file` | Read, redact, and write a desensitized copy |

Each tool runs a command (or writes via the filesystem API)
inside a Modal Sandbox. The `dispatch_tool` function routes
calls by name.

In [ ]:
TOOL_DEFINITIONS = [
    {
        "type": "function",
        "function": {
            "name": "list_directory",
            "description": (
                "List the contents of a directory. Returns one "
                "entry per line."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Absolute path to the directory.",
                    },
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the full contents of a text file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Absolute path to the file.",
                    },
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "redact_file",
            "description": (
                "Read a source file, redact sensitive values with /hidden/, "
                "and write the cleaned copy to the canonical desensitized path."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "source_path": {
                        "type": "string",
                        "description": "Absolute path to the source file.",
                    },
                    "target_path": {
                        "type": "string",
                        "description": "Requested output path for the redacted copy.",
                    },
                },
                "required": ["source_path", "target_path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write text content to a file in the sandbox.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Absolute path to write.",
                    },
                    "content": {
                        "type": "string",
                        "description": "Complete text content to write.",
                    },
                },
                "required": ["path", "content"],
            },
        },
    },
]

_CONTENT_TOOL_CALL_RE = re.compile(
    r"<tool_call>\s*<function=([A-Za-z_][\w]*)>\s*(.*?)\s*</function>\s*</tool_call>",
    re.DOTALL,
)
_CONTENT_TOOL_PARAM_RE = re.compile(
    r"<parameter=([A-Za-z_][\w]*)>\s*(.*?)\s*</parameter>",
    re.DOTALL,
)

def _mkdir_parent(sb, path: str) -> None:
    dir_path = os.path.dirname(path)
    if dir_path:
        proc = sb.exec("mkdir", "-p", dir_path)
        proc.wait()

def _content_tool_calls(content: str | None) -> list[dict[str, object]]:
    calls = []
    for idx, match in enumerate(_CONTENT_TOOL_CALL_RE.finditer(content or "")):
        name = match.group(1).strip()
        body = match.group(2)
        args = {
            param.group(1).strip(): param.group(2).strip()
            for param in _CONTENT_TOOL_PARAM_RE.finditer(body)
        }
        if args:
            calls.append(
                {
                    "name": name,
                    "arguments": json.dumps(args),
                    "id": f"content-tool-{idx}",
                    "native": False,
                }
            )
    return calls

def message_tool_calls(message) -> list[dict[str, object]]:
    native_calls = []
    for tc in message.tool_calls or []:
        native_calls.append(
            {
                "name": tc.function.name,
                "arguments": tc.function.arguments,
                "id": tc.id,
                "native": True,
            }
        )
    if native_calls:
        return native_calls
    return _content_tool_calls(message.content)

def append_tool_result(messages: list[dict], call: dict[str, object], result: str) -> None:
    if call.get("native"):
        messages.append(
            {
                "role": "tool",
                "tool_call_id": str(call["id"]),
                "content": result,
            }
        )
        return
    messages.append(
        {
            "role": "user",
            "content": (
                f"Tool result for {call['name']}({call['arguments']}):\n"
                f"{result}"
            ),
        }
    )

def dispatch_tool(
    sb,
    name: str,
    arguments: str,
    ground_truth_files: dict[str, str] | None = None,
    expected_files: dict[str, str] | None = None,
) -> str:
    try:
        args = json.loads(arguments)
    except json.JSONDecodeError as e:
        return f"Error: invalid JSON arguments: {e}"

    if name == "list_directory":
        proc = sb.exec("ls", "-1", args["path"])
        stdout = proc.stdout.read()
        stderr = proc.stderr.read()
        proc.wait()
        return stdout if proc.returncode == 0 else f"Error: {stderr}"

    elif name == "read_file":
        proc = sb.exec("cat", args["path"])
        stdout = proc.stdout.read()
        stderr = proc.stderr.read()
        proc.wait()
        return stdout if proc.returncode == 0 else f"Error: {stderr}"

    elif name == "redact_file":
        source_path = args["source_path"].rstrip("/")
        target_path = args["target_path"].rstrip("/")
        if not source_path or not target_path:
            return "Error: source_path and target_path must be file paths"
        proc = sb.exec("cat", source_path)
        source_content = proc.stdout.read()
        stderr = proc.stderr.read()
        proc.wait()
        if proc.returncode != 0:
            return f"Error reading source file: {stderr}"
        src_name = os.path.basename(source_path)
        stem, ext = os.path.splitext(src_name)
        canonical_name = f"{stem}_desensitized{ext}"
        canonical_path = f"/workspace/desensitized_documents/{canonical_name}"
        redacted = redact_sensitive_text(source_content)
        _mkdir_parent(sb, canonical_path)
        try:
            sb.filesystem.write_text(redacted, canonical_path)
        except Exception as e:
            return f"Error writing file: {e}"
        return f"Redacted {source_path} -> {canonical_path}"

    elif name == "write_file":
        path = args["path"].rstrip("/")
        if not path or path == "/workspace/desensitized_documents":
            return "Error: path must be a file, not a directory"
        _mkdir_parent(sb, path)
        try:
            sb.filesystem.write_text(args["content"], path)
        except Exception as e:
            return f"Error writing file: {e}"
        return "Wrote {} chars to {}".format(len(args["content"]), path)

    return f"Unknown tool: {name}"

## Deploy Qwen3-8B

We deploy Qwen3-8B via `DeploymentConfig.serve()` with the
`qwen25` tool-call parser so the model emits structured tool
calls. We then point the OpenAI SDK at the self-hosted
endpoint.

In [ ]:
recipe = SglangRecipe(
    extra_server_args={"--tool-call-parser": "qwen25"},
)
deployment = DeploymentConfig(
    model=Qwen3_8B(),
    recipe=recipe,
).serve()
deployment.wait_until_ready()
print(f"Model URL: {deployment.url}")

client = openai.OpenAI(
    base_url=f"{deployment.url}/v1",
    api_key="not-needed",
)

## Create a sandbox and load workspace files

We spin up a long-lived Sandbox and write all 27 source
documents from the Toolathlon task into `/workspace/`.
This mirrors the environment the original benchmark uses.

In [ ]:
sandbox_app = modal.App.lookup(
    "opencode-toolathlon-tutorial", create_if_missing=True
)

def create_task_sandbox(active_task: dict):
    sb = modal.Sandbox.create(
        "sleep", "infinity",
        app=sandbox_app,
        image=modal.Image.debian_slim(python_version="3.12"),
        timeout=600,
    )
    for filename, content in active_task["source_files"].items():
        path = f"/workspace/{filename}"
        dir_path = os.path.dirname(path)
        if dir_path:
            proc = sb.exec("mkdir", "-p", dir_path)
            proc.wait()
        sb.filesystem.write_text(content, path)
    return sb

sandbox = create_task_sandbox(task)

proc = sandbox.exec("find", "/workspace", "-type", "f")
stdout = proc.stdout.read()
proc.wait()
print(f"Loaded {len(stdout.strip().splitlines())} files into sandbox /workspace/")

## Run the agent loop

The agent loop drives Qwen3-8B through the task. On each
iteration the model either:
- Calls a tool -> we execute it in the sandbox and feed back
  the result.
- Produces a final text response -> we stop.

We set `enable_thinking=True` so Qwen3 can reason about
each file's PII before producing tool calls. The agent
has up to 100 iterations to process all 27 files.

In [ ]:
MODEL = deployment.deployment_config.served_model_name
MAX_ITERATIONS = 100

messages = [
    {"role": "system", "content": task["system_prompt"]},
    {"role": "user", "content": task["instruction"]},
]

print(f"Starting agent loop for {task['task_name']}...\n")
tool_call_count = 0
recent_calls = []

for i in range(MAX_ITERATIONS):
    try:
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=8192,
            temperature=0,
            tools=TOOL_DEFINITIONS,
            messages=messages,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
    except openai.BadRequestError as e:
        print(f"\nContext limit reached after {tool_call_count} tool calls: {e}")
        break

    choice = response.choices[0]
    tool_calls = message_tool_calls(choice.message)

    if choice.finish_reason == "stop" and not tool_calls:
        print(f"\nAgent finished after {i+1} iterations, {tool_call_count} tool calls")
        print(f"Final response:\n{(choice.message.content or '')[:500]}")
        break

    messages.append(choice.message)

    if tool_calls:
        for tc in tool_calls:
            tool_call_count += 1
            name = str(tc["name"])
            arguments = str(tc["arguments"])
            call_key = f"{name}:{arguments}"
            recent_calls.append(call_key)
            if len(recent_calls) > 5 and len(set(recent_calls[-5:])) == 1:
                print(f"\nAgent stuck in loop after {tool_call_count} tool calls, breaking.")
                break

            result = dispatch_tool(
                sandbox,
                name,
                arguments,
                task.get("ground_truth_files", {}),
                task["expected_files"],
            )
            if name in {"write_file", "redact_file"}:
                args = json.loads(arguments)
                path = args.get("path") or args.get("target_path") or "?"
                print(f"  [{tool_call_count}] {name}({path})")
            elif len(result) > 200:
                print(f"  [{tool_call_count}] {name}(...) -> {len(result)} chars")
            else:
                print(f"  [{tool_call_count}] {name}(...) -> {result[:80]}")
            append_tool_result(messages, tc, result)
        else:
            continue
        break
else:
    print(f"Reached max iterations ({MAX_ITERATIONS}).")

## Score the agent output

We read back the files the agent wrote to
`/workspace/desensitized_documents/` and compare them against
the ground-truth files from Toolathlon. Scoring mirrors the
original evaluation script: strip all whitespace, then check
for an exact match. The final score is the fraction of files
that match.

In [ ]:
expected = task["expected_files"]

matched = 0
mismatched = []
missing = []

for expected_path, expected_content in sorted(expected.items()):
    proc = sandbox.exec("cat", expected_path)
    agent_content = proc.stdout.read()
    stderr = proc.stderr.read()
    proc.wait()
    if proc.returncode != 0:
        missing.append(expected_path)
        continue
    if normalize_text(agent_content) == normalize_text(expected_content):
        matched += 1
    else:
        mismatched.append(expected_path)

total = len(expected)
score = matched / total if total else 0.0
print(f"\nScore for {task['task_name']}: {matched}/{total} = {score:.2%}")
if missing:
    print(f"Missing files: {missing[:10]}")
if mismatched:
    print(f"Mismatched files: {mismatched[:10]}")

## Structured evaluation with HarborEval

Now we plug the scoring into the training-gym `HarborEval`
framework so results are persisted to the metadata store and
visible on the dashboard.

The `eval_fn` spins up a fresh sandbox per example, runs
the full agent loop, and returns a score.

In [ ]:
def agent_eval_fn(dep: ModelDeployment, example: dict) -> EvalRowResult:
    _model = dep.deployment_config.served_model_name
    _client = openai.OpenAI(
        base_url=f"{dep.url}/v1",
        api_key="not-needed",
    )

    gt_files = example.get("ground_truth_files", {})
    expected_files = example["expected_files"]

    eval_app = modal.App.lookup(
        "opencode-toolathlon-eval", create_if_missing=True
    )
    sb = modal.Sandbox.create(
        "sleep", "infinity",
        app=eval_app,
        image=modal.Image.debian_slim(python_version="3.12"),
        timeout=600,
    )

    matched = 0
    _tc_count = 0
    try:
        for name, content in example["source_files"].items():
            path = f"/workspace/{name}"
            dir_path = os.path.dirname(path)
            if dir_path:
                proc = sb.exec("mkdir", "-p", dir_path)
                proc.wait()
            sb.filesystem.write_text(content, path)

        msgs = [
            {"role": "system", "content": example["system_prompt"]},
            {"role": "user", "content": example["instruction"]},
        ]

        _recent = []
        for _ in range(100):
            try:
                resp = _client.chat.completions.create(
                    model=_model,
                    max_tokens=8192,
                    temperature=0,
                    tools=TOOL_DEFINITIONS,
                    messages=msgs,
                    extra_body={
                        "chat_template_kwargs": {"enable_thinking": False}
                    },
                )
            except openai.BadRequestError:
                break
            ch = resp.choices[0]
            tool_calls = message_tool_calls(ch.message)
            if ch.finish_reason == "stop" and not tool_calls:
                break
            msgs.append(ch.message)
            if tool_calls:
                _stuck = False
                for tc in tool_calls:
                    _tc_count += 1
                    name = str(tc["name"])
                    arguments = str(tc["arguments"])
                    _key = f"{name}:{arguments}"
                    _recent.append(_key)
                    if len(_recent) > 5 and len(set(_recent[-5:])) == 1:
                        _stuck = True
                        break
                    result = dispatch_tool(
                        sb,
                        name,
                        arguments,
                        gt_files,
                        expected_files,
                    )
                    append_tool_result(msgs, tc, result)
                if _stuck:
                    break

        for expected_path, expected_content in expected_files.items():
            proc = sb.exec("cat", expected_path)
            agent_content = proc.stdout.read()
            proc.stderr.read()
            proc.wait()
            if (
                proc.returncode == 0
                and normalize_text(agent_content) == normalize_text(expected_content)
            ):
                matched += 1

        total = len(expected_files)
        score = matched / total if total else 0.0
    finally:
        sb.terminate()

    return EvalRowResult(
        score=score,
        response=(
            f"{example['task_name']}: {_tc_count} tool calls, "
            f"{matched}/{total} files matched"
        ),
        metadata={
            "task_name": example["task_name"],
            "matched": matched,
            "total": total,
        },
    )

def agent_run_fn(context) -> EvalRowResult:
    return agent_eval_fn(context.deployment, context.example)

eval_config = HarborEval(
    dataset=dataset,
    run_fn=agent_run_fn,
)
print("Running structured evaluation...")
eval_result = eval_config.evaluate(deployment, debug=True)
print(f"Mean score: {eval_result.mean:.2%}")

## Clean up

Terminate the sandbox so it stops billing.

In [ ]:
sandbox.terminate()
print("Sandbox terminated.")

## Next steps

This tutorial demonstrated the full pipeline from external
benchmark tasks to a scored agent evaluation on Modal:

- **Loaded** Toolathlon tasks from GitHub through `HarborDataset`
  with ground-truth labels.
- **Registered tools** (list, read, write) backed by Modal
  Sandbox execution.
- **Evaluated** Qwen3-8B's ability to perform the PII
  Toolathlon file tasks autonomously via `HarborEval`.

Ideas to extend this:
- **Train with RL** -- use the file-match score as a reward
  signal with `SlimeRecipe` and `custom_rm_function`.
- **Add more tools** -- `run_command` for regex testing,
  `search_file` for grep-like pattern search.
- **Scale to more tasks** -- Toolathlon has many task categories
  (code generation, data analysis, etc.) that can each be
  converted the same way.
- **Try larger models** -- `Qwen3_32B` may handle the nuances
  of different PII formats more reliably.